In [5]:
library(data.table)
library(tidyverse)
require("ggrepel")
library(pheatmap)
options(repr.plot.width=15, repr.plot.height=8)

# Calling differential Peaks through edgeR with csaw normalization factors

In [6]:
library(edgeR)
library(dplyr)

In [7]:
TMM_normalization_factor <- read.csv('csaw_normalization/efficiency_biases_1_22XY_All_NormalizationFactors.tsv',sep = '\t')
# TMM_normalization_factor <- read.csv('csaw_normalization/CompBias_1_22XY_NormalizationFactors.tsv',sep = '\t')
head(TMM_normalization_factor)

,ID,CellType,Assay,TMM_normalization_factor,lib_size,SizeFactors,SizeFactors.Reciprocal
,<int>,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>
1,390,Glu,Ac,0.9495392,25663115,24.36813,0.04103720
2,4365,Glu,Ac,0.7589804,34983391,26.55171,0.03766236
3,4411,Glu,Ac,0.8577417,23575675,20.22184,0.04945149
4,4414,Glu,Ac,0.7687149,21043413,16.17638,0.06181851
5,4428,Glu,Ac,0.9321073,27768062,25.88281,0.03863568
6,1275,Glu,Ac,1.0797797,19424332,20.97400,0.04767808


In [ ]:
sampleinfo <- read.csv('chip_meta.csv',sep = ',')
sampleinfo <- unique(sampleinfo[c('id','AgeGroup')])
age_groups <- c("Infancy", "Early_Childhood", "Late_Childhood", "Adolescence", "Early_Adulthood",'Adulthood')

## EDGER using CSAW efficiencybias norm factor

In [ ]:
for (assay in  c('Ac','Het','Pol')){
    for (celltype in c('GABA','Glu')){
        df <- fread(paste0('peaks/',celltype,'_',assay,'_peak_raw_counts.tsv.gz'))
        positional_info <- df[, 1:3]
        counts <- df[, -(1:3)]
        colnames(counts) <- sub("^X", "", colnames(counts))
        dge <- DGEList(counts = counts)

        # Filtering TMM normalization factors based on specific conditions
        TMM_normalization_factor_subset <- TMM_normalization_factor[
        TMM_normalization_factor$CellType == celltype & 
        TMM_normalization_factor$Assay == assay, 
        ]

        # Ordering normalization factors according to the column names in DGE object
        ordering <- match(TMM_normalization_factor_subset$ID, colnames(dge))
        TMM_normalization_factor_ordered <- TMM_normalization_factor_subset[order(ordering), ]

        # Assigning library sizes and normalization factors
        dge$samples$lib.size <-( TMM_normalization_factor_ordered$lib_size)
        dge$samples$norm.factors <- TMM_normalization_factor_ordered$TMM_normalization_factor  # Ensure this column name matches your data frame

        # Setting up groups from sample information
        dge$samples$group <- factor(sampleinfo$AgeGroup[match(colnames(dge$counts), sampleinfo$id)])
        
        design <- model.matrix(~0 + group, data = dge$samples)
        colnames(design) <- levels(dge$samples$group)  

        comp_vec = c()
        stage_order <- c('Infancy', 'Early_Childhood', 'Late_Childhood', 'Adolescence', 'Early_Adulthood')
        for (itr in 1:5) {
            itr_nms2 <- stage_order[-(1:itr)]  # Exclude the current stage from comparisons
            itr_nms1 <- rep(stage_order[itr], times=length(itr_nms2))
            comp_vec <- append(comp_vec, paste(itr_nms1, itr_nms2, sep='-'))
        }
        contrasts_matrix <- makeContrasts(contrasts=comp_vec, levels=design)

        
        dge <- estimateDisp(dge, design, robust = TRUE)
        fit <- glmQLFit(dge, design, robust = TRUE)
        results_list <- list()
        for (contrast_name in colnames(contrasts_matrix)) {
            test <- glmQLFTest(fit, contrast = contrasts_matrix[,contrast_name])
            results <- topTags(test, n = Inf)$table
            results <- merge(results, positional_info, by = "row.names", all.x = TRUE)
            significant_results <- results[results$FDR < 0.05, ]
            if (nrow(significant_results) > 0){
            significant_results$AgeGroup1 <- sub("-.*", "", contrast_name)  # Extract first part of contrast name
            significant_results$AgeGroup2 <- sub(".*-", "", contrast_name)  # Extract second part of contrast name
            results_list[[contrast_name]] <- significant_results
            }
        }
        # Combine all results into a single data frame
        combined_results <- bind_rows(results_list, .id = "ContrastName")
        # fwrite(combined_results,paste0('peaks/differential_results_csaw_TMM_EDGER/',celltype,'_',assay,'_differential_results_EDGER.csv.gz'))
    }
}